In [14]:
import gc
import os
import smatch
import numpy as np

import pandas as pd
import matplotlib
import astropy.io.fits as pyfits
import matplotlib.pyplot as plt
from astropy.table import Table,join,vstack

import sim_analysis_utilities as utilities

anaDir='./anaCat'
version =   's19-Hironao-2'
obsDir  =   '../../database' 

In [2]:
cd ./wrkDir/s19/s19-1/

/work/xiangchong.li/work/galSim-HSC/s19/s19-1


# Overview
Prepare Simulation:
+ Prepare field catalogs

+ prepare small catalog

# Make field catalog for simulation

sim_analysis_utilities.prepare_field_catalog

# DataFame Catalog (Simulation)

In [3]:
# Make pd.DataFrame catalog
# The catalog will be used to determine subfield weigthts
# compare 2D histograms, determine optimal weighting, and do calibrations

allSim=[]
for fieldname in utilities.field_names:
    anaFieldDir =   os.path.join(anaDir,'%s' %(fieldname))
    simFname    =   os.path.join(anaFieldDir,'%s-2.fits' %(fieldname)) # -2 is the one with COSMOS redshfit
    allSim.append(utilities.make_smallcat_sim(simFname,fieldname))

/work/xiangchong.li/superonionGW/code/FPFS_Private/FPFSBASE/python/sim_analysis_utilities.py:1059: RuntimeWarning: invalid value encountered in log10
  magnitude   =   -2.5 * np.log10(catalog['base_CircularApertureFlux_3_0_instFlux']) + 27.
/work/xiangchong.li/superonionGW/code/FPFS_Private/FPFSBASE/python/sim_analysis_utilities.py:1108: RuntimeWarning: invalid value encountered in log10
  magnitude   =   -2.5 * np.log10(catalog['base_LocalBackground_instFlux']) + 27.


In [4]:
allSim  =   pd.concat(allSim)
outFname=os.path.join(anaDir,'allSim-2.h5')
allSim.to_hdf(outFname,key='data')

# Plot the 1D and 2D histograms

galDisAnalysis.py

galDisAnalysi2D.py



# get the averaged value of observables
|field/obs |snr| res| mag| absE| noiVar| FWHM|
|----------|---|----|----|-----|-------|-----|
|GAMA09H   |   |    |    |     |       |     |
|GAMA15H   |   |    |    |     |       |     |
|WIDE12H   |   |    |    |     |       |     |
|HECTOMAP  |   |    |    |     |       |     |
|VVDS      |   |    |    |     |       |     |
|XMM       |   |    |    |     |       |     |
|All       |   |    |    |     |       |     |
|----------|---|----|----|-----|-------|-----|


In [23]:
simFname    =   os.path.join(anaDir,'allSim-2.h5')
sim =   pd.read_hdf(simFname,mode='r').to_records(index=True)
obsFname=   os.path.join(obsDir,'smallcat-wl-s19-2-2.fits')
obs=pyfits.getdata(obsFname)

In [24]:
mask    =   np.ones(len(sim),dtype=bool)
mask    &=  (sim['logb']<=-0.375)
mask    &=  (sim['res']>0.3)
mask    &=  (sim['snr']>10.)
mask    &=  (sim['absE']<2.)
mask    &=  ((sim['mag']-sim['a_i']<24.5))
mask    &=  (sim['magA10']<25.5)
sim=sim[mask]

/work/xiangchong.li/lsstsw/miniconda/envs/lsst-scipipe/lib/python3.7/site-packages/ipykernel_launcher.py:7: RuntimeWarning: invalid value encountered in less
  import sys


In [27]:
maskO   =   np.ones(len(obs),dtype=bool)
maskO   &=  (obs['npass']>=2)
maskO   &=  (obs['logb']<=-0.375)
maskO   &=  (obs['res']>0.3)
maskO   &=  (obs['snr']>10.)
maskO   &=  (obs['absE']<2.)
maskO   &=  ((obs['mag']-obs['a_i'])<24.5)
maskO   &=  (obs['magA10']<25.5)
obs=obs[maskO]

In [34]:
for fieldname in utilities.field_names:
    simI=sim[sim['fieldname']==fieldname]
    obsI=obs[obs['fieldname']==fieldname]
    weights=simI['weight']
    print(np.average(simI['mag'],weights=weights),np.average(obsI['mag']))
    break


23.3026727360787 23.159262
